# Iterables, Iterators, and Generators as Runtime Protocols
This notebook has been rewritten to be slower, more reflective, and less template-like. Instead of treating **Iterables, Iterators, and Generators as Runtime Protocols** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


When people struggle with **Iterables, Iterators, and Generators as Runtime Protocols**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- See iteration as a protocol rather than a feature list
- Use `iter` and `next` directly with confidence
- Understand exhaustion
- Connect generators to the same protocol


An iterator is stateful. It needs to remember where it currently is in the sequence or stream. That remembered progress is the key memory-level difference between an iterable container and a one-shot iterator.


In [1]:
items = [10, 20, 30]
it = iter(items)
print(id(it))
print(next(it))
print(next(it))


2713656585536
10
20


Disassembly of a `for` loop shows iteration instructions that drive the protocol under the hood. This is one of the nicest places to see syntax lowering into interpreter steps.


In [2]:
import dis

def walk(values):
    for value in values:
        print(value)

dis.dis(walk)


  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (values)
              4 GET_ITER
        >>    6 FOR_ITER                17 (to 42)
              8 STORE_FAST               1 (value)

  5          10 LOAD_GLOBAL              1 (NULL + print)
             22 LOAD_FAST                1 (value)
             24 PRECALL                  1
             28 CALL                     1
             38 POP_TOP
             40 JUMP_BACKWARD           18 (to 6)

  4     >>   42 LOAD_CONST               0 (None)
             44 RETURN_VALUE


A list can produce many fresh iterators; a generator is often itself the iterator.

Once consumed, they do not reset automatically.

That is why `StopIteration` matters.

They are not a separate world; they are a pleasant way to implement the same contract.


This is what the loop is hiding for your convenience.


In [3]:
data = [1, 2, 3]
it = iter(data)
print(next(it))
print(next(it))
print(next(it))


1
2
3


Once an iterator is finished, asking for more raises StopIteration.


In [4]:
it = iter([1])
print(next(it))
try:
    print(next(it))
except StopIteration:
    print("finished")


1
finished


The same `next` protocol works here as well.


In [5]:
def squares(n):
    for i in range(n):
        yield i * i

g = squares(3)
print(next(g))
print(next(g))
print(next(g))


0
1
4


This is not only a classroom topic. **Iterables, Iterators, and Generators as Runtime Protocols** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Assuming any iterable is also a reusable iterator
- Passing one-shot iterators into code that iterates twice
- Treating generators as mysterious rather than protocol-based


- What is the difference between an iterable and an iterator?
- What does a `for` loop do under the hood?
- Why do generators get exhausted?


- Iteration in Python is protocol-based.
- Iterators are stateful and one-shot by default.
- Generators are a friendly way to implement the protocol.


If this notebook did its job, **Iterables, Iterators, and Generators as Runtime Protocols** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.
